In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/Users/hrchapmandutton/Documents/Projects/CDW/all_fires_2001_2019_perc_single_burn.csv')
df.columns

In [ ]:
df_filter = df[df['MaxData'] == 'yes']
df_filter=df_filter[df_filter['NA_L1NAME']!='NORTH AMERICAN DESERTS']
df_filter=df_filter[df_filter['NA_L1NAME']!='GREAT PLAINS']

In [ ]:
rep_dates = pd.to_datetime(df_filter["REP_DATE"], errors="coerce")
fpout_dates = pd.to_datetime(df_filter["FPOUTDATE"], errors="coerce")
best_dates = rep_dates.fillna(fpout_dates)
existing_month = pd.to_numeric(df_filter["MONTH"], errors="coerce")
existing_day = pd.to_numeric(df_filter["DAY"], errors="coerce")

df_filter["MONTH"] = best_dates.dt.month.fillna(existing_month)
df_filter["DAY"] = best_dates.dt.day.fillna(existing_day)

print(df_filter["MONTH"].unique())

In [ ]:
cats = ['NA_L1NAME', 'NetRF', 'Recovery', 'Albedo', 'Permafrost', 'GHG']
df_RF = df_filter[cats]
df_RF.head()

In [ ]:
cats2 = ['Country', 'NetRF', 'Recovery', 'Albedo', 'Permafrost', 'GHG']
df_RF_states = df_filter[cats2]
df_RF_states.head()

In [ ]:
plt.figure(figsize=(10,8))
ax = sns.violinplot(data=df_RF, x='NA_L1NAME', y='NetRF', hue='PRESCRIBED', split=True, gap=0.1, inner="quartile")

order = [t.get_text() for t in ax.get_xticklabels()]
by_type = (
    df_RF.groupby(["NA_L1NAME", "PRESCRIBED"])
.size()
    .unstack(fill_value=0)
)

import textwrap

labels = []
for name in order:
    y_count = by_type.loc[name, "Y"] if "Y" in by_type.columns and name in by_type.index else 0
    n_count = by_type.loc[name, "N"] if "N" in by_type.columns and name in by_type.index else 0
    wrapped_name = textwrap.fill(name, width=12)
    labels.append(f"{wrapped_name}\n\n[Wildfire={n_count}]\n[Prescribed={y_count}]")

ax.set_xticklabels(labels)

handles, current_labels = ax.get_legend_handles_labels()
label_map = {"N": "Wildfire", "Y": "Prescribed"}
new_labels = [label_map.get(label, label) for label in current_labels]
ax.legend(handles=handles, labels=new_labels, title="Fire Type")

ax.axhline(0, color='gray', linewidth=1, alpha=0.5, zorder=0)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.tick_params(axis='x', labelsize=8)

plt.xlabel('')
plt.ylabel('Mean net RF per fire')
plt.title('Net RF per Single-Burn Fire across Ecoregions')
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
ax = sns.histplot(data=df_RF, x='NetRF', hue='PRESCRIBED', bins=80, multiple="stack")

label_map = {"N": "Wildfire", "Y": "Prescribed"}
ax.legend_.set_title("Fire Type")
for text in ax.legend_.texts:
    text.set_text(label_map.get(text.get_text(), text.get_text()))

plt.xlabel('Mean net RF per Fire')
plt.ylabel('Count of Fires')

plt.show()

In [ ]:
plt.figure(figsize=(8,8))
ax = sns.catplot(data=df_RF, x='MONTH', y='NetRF', hue='PRESCRIBED', capsize=.2, kind='point', errorbar='se')
#ax.axhline(0, color='gray', linewidth=1, alpha=0.5, zorder=0)
plt.show()